# Preset-Katalog für die Website

Der Standardkatalog folgt `pexl.schema.current`. Beim Rendern liest PEXL die Preset-Tabellen direkt aus den Excel-Schemaexporten und erstellt die Website-Fragmente neu. Die Versionsauswahl umfasst die Exporte, die Preset-Tabellen enthalten. Historische Variablen verwenden die Labels und Einheiten ihres jeweiligen Exports. Beim normalen Seitenaufruf erscheint das aktive Schema.

In [1]:
#| code-fold: true
#| code-summary: Vorbereitung für lokale Ausführung
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

import openpyxl

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/output_catalog.py').is_file())
sys.path.insert(0, str(ROOT))
from pexl.schema import current
from pexl.schema._presets import load_preset_catalogs
from scripts.output_catalog import SCHEMA_EXPORTS, build_catalogs

## Excel-Exporte und Versionsauswahl prüfen

Der erste Katalog gehört zum aktiven Schema. Neuere Exporte geben die vollständige Toolversion in `META.tool_version` an. Bei älteren Exporten steht dort zum Teil nur die Versionsreihe. In diesem Fall stammt der Patchstand aus dem Namen des Exports.

In [2]:
catalogs = load_preset_catalogs(SCHEMA_EXPORTS)
active, *historical = catalogs
assert active['version_key'] == current.SCHEMA_VERSION
for data in catalogs:
    count = sum(len(table['presets']) for table in data['tables'])
    print(data['tool_version'], len(data['tables']), count, data['source'].name)

1.13.3 26 285 schema_1.13.3.xlsx
1.13.2_dev 26 285 schema_1.13.2_dev.xlsx
1.13.1_dev 26 285 schema_1.13.1_dev.xlsx
v1.12.7 26 242 schema_v1_12_7.xlsx
v1.11.6 21 232 schema_v1_11_6.xlsx


## Werte gegen Excel abgleichen

Für jeden Stand werden die exportierten Werte mit ihren Quellzellen verglichen. Damit bleiben auch Null, Falsch und leere Zellen unterscheidbar.

In [3]:
for data in catalogs:
    workbook = openpyxl.load_workbook(data['source'], data_only=True)
    try:
        for table in data['tables']:
            sheet = workbook[table['name']]
            for preset in table['presets']:
                for key, address in preset['cells'].items():
                    actual, expected = sheet[address].value, preset['values'][key]
                    assert type(actual) is type(expected) and actual == expected, (data['tool_version'], table['name'], address)
    finally:
        workbook.close()
print('Preset-Werte gegen die Excel-Quellzellen abgeglichen.')

Preset-Werte gegen die Excel-Quellzellen abgeglichen.


## Website-Fragmente erzeugen

Der Test verwendet denselben Generator wie Quarto und schreibt in einen temporären Ordner. Das Hauptfragment enthält den aktiven Katalog. Die Website lädt die historischen Fragmente erst bei Auswahl.

In [4]:
with TemporaryDirectory() as temporary:
    output = Path(temporary)
    in_catalog, out_catalog, preset_catalog = build_catalogs(output)
    assert in_catalog.is_file() and out_catalog.is_file()
    html = preset_catalog.read_text(encoding='utf-8')
    assert f'data-current="{current.SCHEMA_VERSION}"' in html
    assert f'Stand: {active["tool_version"]}' in html
    for data in historical:
        fragment = output / 'preset-versions' / f'{data["version_key"]}.html'
        assert f'Stand: {data["tool_version"]}' in fragment.read_text(encoding='utf-8')
print('Aktiver Katalog und historische Website-Fragmente geprüft.')

Aktiver Katalog und historische Website-Fragmente geprüft.


## Schemawechsel und Vorschau

Einen neuen Excel-Stand wie im [Schema-How-to](howto_schema_transition.ipynb) exportieren und aktivieren. Anschließend vom Repository aus mit der aktiven Projektumgebung `quarto render website` ausführen. Quarto erzeugt die Kataloge vor dem Rendern. Fehlt der zum aktiven Schema passende Export, meldet der Build die erwartete Version.

In der Vorschau unter `/preset-catalog.html` und `/en/preset-catalog.html` die Standardversion, den Wechsel auf einen früheren Excel-Stand, die Suche und die Rückkehr zur aktiven Version prüfen. Die URL enthält bei historischen Katalogen die gewählte Version und kann geteilt werden. Beispiele und Fachseiten-Verweise für den aktuellen Stand stehen in `website/_preset-notes.yml`.